In [1]:
# General imports
import os
from netsim.netSimPy import *
from netsim.allocators import sap_ff, ldpb, sap_ff2

### Estrategia de prueba
1. Primero se crea una red con igual prbabilidad de llegada y de salidas
2. Se atienden 1.000.000 solicitudes de conexión y se valida que efectivamente no se generan conexiones que quedan encoladas (como pensando en que esta malo el simulador y deja varios FSUs ocupados cuando se elimina la conexión) En este caso solo quedan 2 conexiones luego de atenderlas todas
3. Luego se valida que efectivamente, en toda la red, no exista ningun slot ocupado que no corresponda a esas dos conexiones.

In [5]:
M_LAMBDA = 100
network = Network(
    networkFileName =  "../../networks/nsfnet/network.json",
    pathsFileName= "../../networks/nsfnet/routes.json",
    bitrateFilename= "../../networks/nsfnet/bitrates_4_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)

sim_args = dict(
    network = network,
    eventsGenerator=generator,
    allocator = sap_ff2(3),
    n_paths=3,
)

simulator = Simulator(**sim_args)
simulator.run(400000)
print(simulator.getNetwork().getAllConnections())

Blocking probability:0.0
{UUID('1462bd09-73cf-45a2-b686-14638d5e5561'): <netsim.netSimPy.connection.Connection object at 0x106e20b60>, UUID('2d805994-20ef-4a08-a652-b03c3d82b8b2'): <netsim.netSimPy.connection.Connection object at 0x1088220c0>}


In [9]:
links = simulator.getNetwork().links
paths = simulator.getNetwork().paths

cons = list(simulator.getNetwork().getAllConnections().values())
for con in cons:
    for linkID in con.getConnectionInfo().keys():
        linkSlots = con.getLinkSlots(linkID)
        print("-------------------------")
        print("Fuente destino: ",con.src, con.dst)
        print("Banda: ",con.getBand(linkID))
        print("ID de enlace: ", linkID, ", Slots: ", linkSlots)
        print("Ruta: ", paths[con.src,con.dst])


-------------------------
Fuente destino:  13 10
Banda:  C
ID de enlace:  13-11 , Slots:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Ruta:  [[13, 11, 10], [13, 12, 10], [13, 12, 8, 11, 10]]
-------------------------
Fuente destino:  13 10
Banda:  C
ID de enlace:  11-10 , Slots:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Ruta:  [[13, 11, 10], [13, 12, 10], [13, 12, 8, 11, 10]]
-------------------------
Fuente destino:  11 3
Banda:  C
ID de enlace:  11-10 , Slots:  [11, 12, 13, 14, 15, 16]
Ruta:  [[11, 10, 3], [11, 8, 7, 6, 4, 3], [11, 13, 12, 10, 3]]
-------------------------
Fuente destino:  11 3
Banda:  C
ID de enlace:  10-3 , Slots:  [11, 12, 13, 14, 15, 16]
Ruta:  [[11, 10, 3], [11, 8, 7, 6, 4, 3], [11, 13, 12, 10, 3]]


In [ ]:
count = 0
for band in simulator.getNetwork().getBands():
    for link in links.values():
        for slot in link.getSlots(band):
            if slot is True:
                # print(slot)
                count+=1
print(count)

In [ ]:
# M_LAMBDA = 50000
# network = Network(
#     networkFileName =  "../../networks/nsfnet/network.json",
#     pathsFileName= "../../networks/nsfnet/routes.json",
#     bitrateFilename= "../../networks/nsfnet/bitrates_4_bands.json",
# )
# generator = EventsGenerator(mLambda=M_LAMBDA)

# sim_args = dict(
#     network = network,
#     eventsGenerator=generator,
#     allocator = sap_ff2(3),
#     n_paths=3,
# )

# simulator = Simulator(**sim_args)
# simulator.run(90000)